In [1]:
# =========================================================
# LB PUSH PIPELINE (ConvNeXt-Large + EMA + 3-STAGE TRAIN)
# Target: 43 → 45+%
# =========================================================

import os, random, glob, time, copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models

from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
from torch.amp import autocast, GradScaler

# ================= CONFIG =================
SEED = 42
BATCH_SIZE = 8
ACCUM_STEPS = 4
N_FOLDS = 3

HEAD_LR = 2e-4
BLOCK4_LR = 8e-5
BLOCK3_LR = 3e-5

WEIGHT_DECAY = 0.05

TIME_LIMIT = 8 * 3600
START_TIME = time.time()

DEVICE = torch.device("cuda")

ROOT = "/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors"
TRAIN_DIR = os.path.join(ROOT, "train")
TEST_DIR = os.path.join(ROOT, "test")

# ================= SEED =================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

# ================= EMA (IMPORTANT BOOST) =================
class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {n: p.clone().detach() for n,p in model.named_parameters() if p.requires_grad}

    def update(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = self.decay*self.shadow[n] + (1-self.decay)*p.data

    def apply(self):
        self.backup = {}
        for n,p in self.model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])

    def restore(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])

# ================= AUG =================
def get_tfms(sz, train=True, stage=1):
    mean = [0.485,0.456,0.406]
    std = [0.229,0.224,0.225]

    if train:
        if stage == 1:
            return transforms.Compose([
                transforms.RandomResizedCrop(sz, scale=(0.9,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.1,0.1,0.1,0.05),
                transforms.ToTensor(),
                transforms.Normalize(mean,std),
            ])
        elif stage == 2:
            return transforms.Compose([
                transforms.RandomResizedCrop(sz, scale=(0.7,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandAugment(2,7),
                transforms.ToTensor(),
                transforms.Normalize(mean,std),
            ])
        else:
            return transforms.Compose([
                transforms.RandomResizedCrop(sz, scale=(0.6,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandAugment(3,9),
                transforms.ToTensor(),
                transforms.Normalize(mean,std),
            ])
    else:
        return transforms.Compose([
            transforms.Resize(int(sz*1.15)),
            transforms.CenterCrop(sz),
            transforms.ToTensor(),
            transforms.Normalize(mean,std),
        ])

# ================= MIXUP =================
def mixup(x,y,alpha=0.2,prob=0.6):
    if np.random.rand() < prob:
        lam = np.random.beta(alpha,alpha)
        idx = torch.randperm(x.size(0)).to(x.device)
        return lam*x + (1-lam)*x[idx], y, y[idx], lam
    return x,y,y,1.0

# ================= MODEL =================
class Model(nn.Module):
    def __init__(self,n):
        super().__init__()
        base = models.convnext_large(weights=models.ConvNeXt_Large_Weights.IMAGENET1K_V1)

        self.features = base.features
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1536, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, n)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.avgpool(x)
        return self.head(x)

# ================= DATA =================
full_ds = datasets.ImageFolder(TRAIN_DIR)
num_classes = len(full_ds.classes)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
test_paths = sorted(glob.glob(os.path.join(TEST_DIR,"*.*")))

all_probs = []

# ================= TRAIN =================
for fold,(t_idx,v_idx) in enumerate(skf.split(np.zeros(len(full_ds)), full_ds.targets)):

    if time.time()-START_TIME > TIME_LIMIT:
        break

    print(f"\n==== FOLD {fold+1} ====")

    model = Model(num_classes).to(DEVICE)
    scaler = GradScaler("cuda")
    ema = EMA(model)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # ================= STAGE 1 =================
    for p in model.parameters(): p.requires_grad = False
    for p in model.head.parameters(): p.requires_grad = True

    optimizer = torch.optim.AdamW(model.head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)

    train_loader = DataLoader(
        Subset(datasets.ImageFolder(TRAIN_DIR, get_tfms(224, True, 1)), t_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2
    )

    for epoch in range(4):
        model.train()
        for x,y in tqdm(train_loader, leave=False):
            x,y = x.to(DEVICE), y.to(DEVICE)
            x,y1,y2,lam = mixup(x,y,alpha=0.1,prob=0.3)

            with autocast("cuda"):
                out = model(x)
                loss = lam*criterion(out,y1)+(1-lam)*criterion(out,y2)

            scaler.scale(loss/ACCUM_STEPS).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    # ================= STAGE 2 =================
    for p in model.features[3:].parameters():
        p.requires_grad = True

    optimizer = torch.optim.AdamW([
        {"params": model.features[3:].parameters(), "lr": BLOCK4_LR},
        {"params": model.head.parameters(), "lr": HEAD_LR}
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)

    train_loader = DataLoader(
        Subset(datasets.ImageFolder(TRAIN_DIR, get_tfms(288, True, 2)), t_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2
    )

    val_loader = DataLoader(
        Subset(datasets.ImageFolder(TRAIN_DIR, get_tfms(288, False)), v_idx),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    best_acc = 0
    best_state = None

    for epoch in range(6):
        model.train()

        for x,y in tqdm(train_loader, leave=False):
            x,y = x.to(DEVICE), y.to(DEVICE)
            x,y1,y2,lam = mixup(x,y,alpha=0.3,prob=0.6)

            with autocast("cuda"):
                out = model(x)
                loss = lam*criterion(out,y1)+(1-lam)*criterion(out,y2)

            scaler.scale(loss/ACCUM_STEPS).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            ema.update()

        scheduler.step()

        # validation EMA
        ema.apply()
        model.eval()

        correct,total=0,0
        with torch.no_grad():
            for x,y in val_loader:
                x,y=x.to(DEVICE),y.to(DEVICE)
                out=model(x)
                correct+=(out.argmax(1)==y).sum().item()
                total+=y.size(0)

        acc=correct/total
        print("ACC:",acc)

        if acc>best_acc:
            best_acc=acc
            best_state=copy.deepcopy(model.state_dict())

        ema.restore()

    model.load_state_dict(best_state)

    # ================= TTA =================
    model.eval()
    fold_probs=[]

    tf1=get_tfms(288,False)
    tf2=transforms.Compose([
        transforms.Resize(340),
        transforms.CenterCrop(288),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    with torch.no_grad():
        for p in tqdm(test_paths,leave=False):
            try:
                img=Image.open(p).convert("RGB")

                x1=tf1(img).unsqueeze(0).to(DEVICE)
                x2=torch.flip(x1,[3])
                x3=tf2(img).unsqueeze(0).to(DEVICE)
                x4=torch.flip(x3,[3])

                out=(model(x1)+model(x2)+model(x3)+model(x4))/4
                fold_probs.append(torch.softmax(out,1).cpu().numpy())

            except:
                fold_probs.append(np.zeros((1,num_classes)))

    all_probs.append(np.vstack(fold_probs))

# ================= SUBMIT =================
final_probs=np.mean(all_probs,axis=0)
final_probs=final_probs**1.15
final_probs/=final_probs.sum(axis=1,keepdims=True)

labels=final_probs.argmax(1)

pd.DataFrame({
    "ImageName":[os.path.basename(p) for p in test_paths],
    "label":labels
}).to_csv("submission.csv",index=False)

print("DONE")


==== FOLD 1 ====
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 223MB/s]


ACC: 0.4469006381039198


ACC: 0.4977210574293528


ACC: 0.5050136736554239


...

ACC: 0.49692272623660816


ACC: 0.4957829952131297


ACC: 0.48894460907225895


DONE


/tmp/ipykernel_23/1530753233.py:292: RuntimeWarning: invalid value encountered in divide
  final_probs/=final_probs.sum(axis=1,keepdims=True)
